# GNN Reranking Server

Serves a trained ResidualGNN model as a FastAPI endpoint for ICD→HPO candidate reranking.

The GNN refines Qwen embeddings with ontology hierarchy context (GAT message passing),
then reranks HPO candidates by cosine similarity in the refined embedding space.

**Prerequisites:**
- `ontology_graph.pkl` and `gnn_checkpoint.pt` in Google Drive
- ngrok domain configured in Colab secrets as `NGROK_GNN_DOMAIN`

In [1]:
%%capture
!pip install -qq fastapi uvicorn pyngrok
!pip install -qq torch
!pip install -qq torch_geometric

In [2]:
from __future__ import annotations

import os
import pickle
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

from pyngrok import ngrok
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from google.colab import userdata
import uvicorn

In [3]:
NGROK_TOKEN = userdata.get("NGROK_TOKEN")
NGROK_GNN_DOMAIN = userdata.get("NGROK_GNN_DOMAIN")

os.environ["NGROK_TOKEN"] = NGROK_TOKEN
!ngrok config add-authtoken "$NGROK_TOKEN"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [4]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/main/projects/presentations/graph-med/")

DATA_PATH       = DRIVE_DIR / "ontology_graph.pkl"
CHECKPOINT_PATH = DRIVE_DIR / "gnn_checkpoint.pt"

assert DATA_PATH.exists(), f"Not found: {DATA_PATH}"
print(f"Drive mounted. Data dir: {DRIVE_DIR}")
print(f"  ontology_graph.pkl : {'OK' if DATA_PATH.exists() else 'MISSING'}")
print(f"  gnn_checkpoint.pt  : {'OK' if CHECKPOINT_PATH.exists() else 'MISSING'}")

API_HOST = "0.0.0.0"
API_PORT = 8002
HIDDEN   = 1024

Mounted at /content/drive
Drive mounted. Data dir: /content/drive/MyDrive/main/projects/presentations/graph-med
  ontology_graph.pkl : OK
  gnn_checkpoint.pt  : OK


# Model Definition

In [5]:
class ResidualGNN(nn.Module):
    """
    GAT that refines original embeddings via a residual connection.

    Architecture:
        input_proj: Linear(in_channels -> hidden)
        convs:      GATConv x num_layers
        output_proj: MLP(hidden -> hidden -> in_channels)
        residual:   output = original_embedding + alpha * output_proj(gnn_out)

    The residual design preserves the strong Qwen baseline by construction.
    """

    def __init__(
        self,
        in_channels: int = 3584,
        hidden_channels: int = 1024,
        num_layers: int = 2,
        heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.num_layers = num_layers
        self.dropout = dropout

        self.input_proj = nn.Linear(in_channels, hidden_channels)

        self.convs = nn.ModuleList()
        for i in range(num_layers):
            if i == num_layers - 1:
                self.convs.append(
                    GATConv(hidden_channels, hidden_channels, heads=1,
                            concat=False, dropout=dropout)
                )
            else:
                self.convs.append(
                    GATConv(hidden_channels, hidden_channels // heads,
                            heads=heads, concat=True, dropout=dropout)
                )

        self.output_proj = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, in_channels),
        )

        # Learnable residual scale (starts small so initial output ~ baseline)
        self.residual_scale = nn.Parameter(torch.tensor(0.1))

    def encode(self, x, edge_index):
        """Returns refined embeddings: original + scale * gnn_correction."""
        h = self.input_proj(x)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index)
            if i < self.num_layers - 1:
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)

        correction = self.output_proj(h)
        return x + self.residual_scale * correction


print(f"Model class defined: ResidualGNN")
print(f"  Hidden: {HIDDEN}, Output: 3584 (same as input via residual)")

Model class defined: ResidualGNN
  Hidden: 1024, Output: 3584 (same as input via residual)


# Load Graph + Checkpoint

In [6]:
# Load exported graph from Drive
with open(DATA_PATH, "rb") as f:
    graph_data = pickle.load(f)

raw_x = torch.tensor(graph_data["x"], dtype=torch.float32)
edge_index = torch.stack([
    torch.tensor(graph_data["edge_src"], dtype=torch.long),
    torch.tensor(graph_data["edge_tgt"], dtype=torch.long),
])
node_info = graph_data["node_info"]
code_to_idx = {n["code"]: i for i, n in enumerate(node_info)}

print(f"Graph loaded: {raw_x.shape[0]} nodes, {edge_index.shape[1]} edges")
print(f"Embedding dim: {raw_x.shape[1]}")

# Load trained model from Drive
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
model = ResidualGNN(in_channels=raw_x.shape[1], hidden_channels=HIDDEN)
state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint["model_state_dict"].items()}
model.load_state_dict(state_dict)
model.eval()

alpha = model.residual_scale.item()
print(f"Checkpoint loaded. Residual scale alpha = {alpha:.4f}")
if "hyperparams" in checkpoint:
    print(f"  Hyperparams: {checkpoint['hyperparams']}")

# Precompute all refined embeddings (one forward pass)
with torch.no_grad():
    refined_emb = model.encode(raw_x, edge_index)
    refined_norm = F.normalize(refined_emb, dim=1)
    raw_norm = F.normalize(raw_x, dim=1)

print(f"Refined embeddings precomputed: {refined_norm.shape}")

Graph loaded: 45205 nodes, 330488 edges
Embedding dim: 3584
Checkpoint loaded. Residual scale alpha = 0.1020
  Hyperparams: {'in_channels': 3584, 'hidden_channels': 1024, 'num_layers': 2}
Refined embeddings precomputed: torch.Size([45205, 3584])


# FastAPI Application

In [7]:
# --- Pydantic models ---

class RerankRequest(BaseModel):
    icd_code: str
    candidate_codes: List[str]
    top_k: Optional[int] = None


# --- App ---

app = FastAPI(title="GNN Reranking Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/healthz")
def healthz():
    return {
        "status": "ok",
        "model": "ResidualGNN",
        "hidden": HIDDEN,
        "alpha": round(alpha, 4),
        "nodes": raw_x.shape[0],
        "edges": edge_index.shape[1],
    }


def _rerank(icd_code: str, candidate_codes: list[str], top_k: int | None):
    """Core reranking logic shared by endpoints."""
    if icd_code not in code_to_idx:
        return None, f"ICD code '{icd_code}' not found in graph"

    icd_idx = code_to_idx[icd_code]

    # Filter to candidates that exist in the graph
    valid = [(c, code_to_idx[c]) for c in candidate_codes if c in code_to_idx]
    if not valid:
        return None, "No valid candidate codes found in graph"

    codes, indices = zip(*valid)
    indices_t = torch.tensor(indices, dtype=torch.long)

    # GNN-refined cosine similarity
    gnn_query = refined_norm[icd_idx].unsqueeze(0)
    gnn_cands = refined_norm[indices_t]
    gnn_scores = (gnn_query @ gnn_cands.T).squeeze(0).tolist()

    # Qwen baseline cosine similarity
    raw_query = raw_norm[icd_idx].unsqueeze(0)
    raw_cands = raw_norm[indices_t]
    qwen_scores = (raw_query @ raw_cands.T).squeeze(0).tolist()

    # Build result list sorted by GNN score
    results = []
    for i, code in enumerate(codes):
        results.append({
            "code": code,
            "label": node_info[code_to_idx[code]]["name"],
            "gnn_score": round(gnn_scores[i], 6),
            "qwen_score": round(qwen_scores[i], 6),
        })

    # Compute ranks (1-based)
    by_gnn = sorted(results, key=lambda x: x["gnn_score"], reverse=True)
    by_qwen = sorted(results, key=lambda x: x["qwen_score"], reverse=True)

    gnn_rank = {r["code"]: i + 1 for i, r in enumerate(by_gnn)}
    qwen_rank = {r["code"]: i + 1 for i, r in enumerate(by_qwen)}

    for r in by_gnn:
        r["gnn_rank"] = gnn_rank[r["code"]]
        r["qwen_rank"] = qwen_rank[r["code"]]
        r["rank_delta"] = r["qwen_rank"] - r["gnn_rank"]  # positive = GNN improved

    if top_k:
        by_gnn = by_gnn[:top_k]

    return by_gnn, None


@app.post("/rerank")
def rerank(req: RerankRequest):
    results, error = _rerank(req.icd_code, req.candidate_codes, req.top_k)
    if error:
        return JSONResponse({"error": error}, status_code=400)
    return JSONResponse({
        "icd_code": req.icd_code,
        "icd_label": node_info[code_to_idx[req.icd_code]]["name"],
        "candidates": results,
    })


@app.post("/compare")
def compare(req: RerankRequest):
    results, error = _rerank(req.icd_code, req.candidate_codes, req.top_k)
    if error:
        return JSONResponse({"error": error}, status_code=400)

    # Also return Qwen-ordered view
    by_qwen = sorted(results, key=lambda x: x["qwen_rank"])

    return JSONResponse({
        "icd_code": req.icd_code,
        "icd_label": node_info[code_to_idx[req.icd_code]]["name"],
        "by_gnn": results,
        "by_qwen": by_qwen,
    })


print("FastAPI app defined with endpoints: /healthz, /rerank, /compare")

FastAPI app defined with endpoints: /healthz, /rerank, /compare


# Serve with ngrok

In [ ]:
public_url = ngrok.connect(API_PORT, "http", domain=NGROK_GNN_DOMAIN).public_url
print(f"Public endpoint: {public_url}")

config = uvicorn.Config(app, host=API_HOST, port=API_PORT, log_level="info")
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [157]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)


Public endpoint: https://gnn-nodes2026.ngrok.io
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "GET /healthz HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "GET /healthz HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /rerank HTTP/1.1" 200 OK
INFO:     2a01:e11:540